# 16. Reproducible scientific evaluators and numerical contracts

![Reproducible evaluator pipeline](../images/16_reproducible_scientific_evaluators.svg)

**Learning goals:** distinguish layers of reproducibility, construct frozen objects that own read-only arrays, validate cross-field invariants, state typed policy families, remove irrelevant row-order variation, publish artifacts atomically, and test failure paths.

In [ ]:
from collections.abc import Mapping, Sequence
from dataclasses import dataclass, field
import json
import os
from pathlib import Path
import tempfile
from typing import Literal

import numpy as np

SEED = 16
rng = np.random.default_rng(SEED)
print(f"NumPy {np.__version__}; seed={SEED}")

## 1. Frozen dataclasses are shallow

`frozen=True` blocks attribute rebinding but does not freeze memory referenced by an array. The caller can mutate a retained alias unless the fitted object takes ownership.

In [ ]:
@dataclass(frozen=True)
class UnsafeFit:
    coefficients: np.ndarray

caller_array = np.array([[1.0], [2.0]])
unsafe = UnsafeFit(caller_array)
caller_array[0, 0] = 99.0
assert unsafe.coefficients[0, 0] == 99.0
print("shallow freeze still aliases caller memory")

## 2. Own, validate, and freeze fitted arrays

The safe fit copies every durable parameter, validates shape and finiteness, marks arrays read-only, and installs them with `object.__setattr__` during `__post_init__`. `compare=False` avoids generated elementwise array equality.

In [ ]:
def readonly_array(value, *, ndim, label):
    array = np.array(value, dtype=np.float64, copy=True)
    if array.ndim != ndim or array.size == 0:
        raise ValueError(f"{label} must be a nonempty {ndim}-D array")
    if not np.isfinite(array).all():
        raise FloatingPointError(f"{label} contains non-finite values")
    array.setflags(write=False)
    return array

Normalization = Literal["raw", "standardized"]

@dataclass(frozen=True)
class LinearFit:
    coefficients: np.ndarray = field(repr=False, compare=False)
    intercept: np.ndarray = field(repr=False, compare=False)
    feature_mean: np.ndarray = field(repr=False, compare=False)
    feature_scale: np.ndarray = field(repr=False, compare=False)
    method: Normalization = "standardized"

    def __post_init__(self):
        weights = readonly_array(self.coefficients, ndim=2, label="coefficients")
        bias = readonly_array(self.intercept, ndim=1, label="intercept")
        mean = readonly_array(self.feature_mean, ndim=1, label="feature_mean")
        scale = readonly_array(self.feature_scale, ndim=1, label="feature_scale")
        if weights.shape[0] != len(mean) or scale.shape != mean.shape:
            raise ValueError("input dimensions are inconsistent")
        if weights.shape[1] != len(bias) or np.any(scale <= 0):
            raise ValueError("output dimensions or scales are inconsistent")
        for name, value in (("coefficients", weights), ("intercept", bias),
                            ("feature_mean", mean), ("feature_scale", scale)):
            object.__setattr__(self, name, value)

    def transform(self, rows: Sequence[Sequence[float]] | np.ndarray) -> np.ndarray:
        matrix = np.asarray(rows, dtype=np.float64)
        if matrix.ndim != 2 or matrix.shape[1] != len(self.feature_mean):
            raise ValueError("rows have an unexpected shape")
        if not np.isfinite(matrix).all():
            raise FloatingPointError("rows contain non-finite values")
        standardized = (matrix - self.feature_mean) / self.feature_scale
        return np.einsum("nd,dk->nk", standardized, self.coefficients, optimize=False) + self.intercept

    def diagnostics(self) -> Mapping[str, object]:
        return {"method": self.method, "input_width": len(self.feature_mean),
                "output_width": len(self.intercept)}

## 3. Test both aliasing directions

Copying prevents the caller's original array from changing the fit. The write flag prevents mutation through the stored parameter. Transform results remain caller-owned new arrays.

In [ ]:
source_weights = np.array([[1.0, -1.0], [0.5, 2.0]])
fit = LinearFit(source_weights, np.zeros(2), np.zeros(2), np.ones(2))
source_weights[0, 0] = 500.0
assert fit.coefficients[0, 0] == 1.0
try:
    fit.coefficients[0, 0] = 3.0
except ValueError:
    mutation_rejected = True
else:
    mutation_rejected = False
result = fit.transform([[1.0, 2.0]])
result[0, 0] = -20.0
assert mutation_rejected and fit.coefficients[0, 0] == 1.0
assert fit.diagnostics() == {"method": "standardized", "input_width": 2, "output_width": 2}
print("caller alias and stored-array mutation are blocked")

## 4. Canonical row order for deterministic fitting

Floating-point reductions can depend on input order. Sort paired rows and targets by exact `float.hex()` keys before fitting. Scientific identifiers would precede numeric values in a real table.

In [ ]:
def fit_multioutput_ridge(rows, targets, alpha=1.0):
    x = np.asarray(rows, dtype=np.float64)
    y = np.asarray(targets, dtype=np.float64)
    if x.ndim != 2 or y.ndim != 2 or len(x) != len(y):
        raise ValueError("rows and targets must be aligned matrices")
    order = sorted(
        range(len(x)),
        key=lambda index: (
            *(float(value).hex() for value in x[index]),
            *(float(value).hex() for value in y[index]),
        ),
    )
    x, y = x[np.asarray(order)], y[np.asarray(order)]
    mean = x.mean(axis=0, dtype=np.float64)
    scale = np.maximum(x.std(axis=0, ddof=0, dtype=np.float64), 1e-12)
    standardized = (x - mean) / scale
    x_mean = standardized.mean(axis=0, dtype=np.float64)
    y_mean = y.mean(axis=0, dtype=np.float64)
    centered_x, centered_y = standardized - x_mean, y - y_mean
    gram = np.einsum("ni,nj->ij", centered_x, centered_x, dtype=np.float64)
    rhs = np.einsum("ni,nk->ik", centered_x, centered_y, dtype=np.float64)
    weights = np.linalg.solve(gram + alpha * np.eye(x.shape[1]), rhs)
    intercept = y_mean - x_mean @ weights
    return LinearFit(weights, intercept, mean, scale)

x = rng.normal(size=(80, 4))
y = x @ rng.normal(size=(4, 2)) + rng.normal(0, .03, size=(80, 2))
permutation = rng.permutation(len(x))
fit_a = fit_multioutput_ridge(x, y)
fit_b = fit_multioutput_ridge(x[permutation], y[permutation])
assert np.array_equal(fit_a.coefficients, fit_b.coefficients)
assert np.array_equal(fit_a.intercept, fit_b.intercept)
assert np.array_equal(fit_a.transform(x), fit_b.transform(x))
print("row permutation preserves fitted bytes")

## 5. Atomic publication and failure injection

Write a same-directory temporary artifact, validate it, and replace the destination only after success. A forced failure must leave the old destination intact and clean up the temporary file.

In [ ]:
def atomic_json(payload, destination, *, fail_before_replace=False):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_name = tempfile.mkstemp(
        prefix=f".{destination.name}.", suffix=".tmp", dir=destination.parent
    )
    os.close(descriptor)
    temporary = Path(temporary_name)
    try:
        temporary.write_text(json.dumps(payload, sort_keys=True) + "\n")
        parsed = json.loads(temporary.read_text())
        if parsed != payload:
            raise ValueError("read-back validation failed")
        if fail_before_replace:
            raise RuntimeError("injected failure")
        os.replace(temporary, destination)
    finally:
        temporary.unlink(missing_ok=True)

with tempfile.TemporaryDirectory() as directory:
    destination = Path(directory) / "result.json"
    destination.write_text('{"generation": 1}\n')
    old_bytes = destination.read_bytes()
    try:
        atomic_json({"generation": 2}, destination, fail_before_replace=True)
    except RuntimeError:
        pass
    assert destination.read_bytes() == old_bytes
    assert not list(destination.parent.glob(f".{destination.name}.*.tmp"))
    atomic_json({"generation": 2}, destination)
    assert json.loads(destination.read_text()) == {"generation": 2}
print("failure preserves the old artifact; success publishes the new artifact")

## Exercises and takeaways

1. Replace `np.array(..., copy=True)` with `np.asarray`. Construct a caller alias and demonstrate the regression.
2. Remove `compare=False` from an array field and inspect generated dataclass equality.
3. Add a participant identifier to the canonical row key ahead of numeric features.
4. Change an exact byte assertion to `np.allclose`. State which contract is weakened.
5. Inject a parse failure into the atomic writer and verify cleanup.
6. Extend the writer to publish a digest in a metadata sidecar. Explain why replacing two files is not one atomic transaction.

**Takeaways:** mathematical formulas need software contracts. Fitted objects should own read-only state, validation should establish relationships at construction, numeric ordering and random streams should be explicit, comparisons should match their purpose, and artifact publication should preserve the last complete result across failures.

## Continue learning

[Previous notebook: 15](15_exposure_and_replication.ipynb) | [Lecture](../lectures/16_reproducible_scientific_evaluators.md) | [Curriculum](../README.md)